# Paper-completion experiments

The closing battery that takes the paper from complete to fully defended.
- **P1 Single-token / prefix robustness** (must-run): per tokenizer, fraction of single-token and
  prefix-unique gold answers, and the dissociation restricted to each. Protects the fact-specific claim.
- **P6 Hard-decoy headline**: readable under mean vs max decoy, with not-top rate under both -- the
  conservative headline numbers.
- **P3 Calibrated dose-response**: explicit targets (25/50/75th pct + mean of successful-item answer
  projection) and recovery curves for answer-up / alternative-down / both / random.
- **P8 Sequence-level sanity**: does first-token recovery (answer-up patch during generation) yield the
  correct FULL answer under greedy decoding? Shows the first-token analysis is not token-local.
- **P2 OLMo trajectory** (separate cell): norm-freq, sign-stable directional-freq, baseline-logit-freq,
  and spectral anisotropy across checkpoints. The two-effect phrasing: co-evolution builds
  directionality (early, flat); weight decay suppresses norm-frequency coupling (decays).

Needs `rw_core.py`, `mech_core.py`, `mech_runner.py`, `olmo_runner.py`. Run their tests first.

## 0. Config + data

In [ ]:
import numpy as np, json, gc, torch, re
import rw_core as rw, mech_core as mc, mech_runner as mr, olmo_runner as olr
from datasets import load_dataset
from collections import defaultdict
import pandas as pd

DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DTYPE=torch.float16 if DEVICE=="cuda" else torch.float32
QA="Answer with a short factual answer.\nQuestion: {q}\nAnswer:"
TEMPLATES=[QA,"Q: {q}\nA:","Please answer concisely.\n{q}\nAnswer:","{q} The answer is"]
N_ITEMS=300
CAPS=dict(single=400, hard=400, dose=200, seq=120)

MODELS=[
 "meta-llama/Llama-3.1-8B","meta-llama/Llama-3.2-3B","meta-llama/Llama-3.2-1B",
 "meta-llama/Llama-3.2-3B-Instruct","Qwen/Qwen2.5-3B","Qwen/Qwen2.5-3B-Instruct",
 "Qwen/Qwen2.5-7B","mistralai/Mistral-7B-v0.1",
]

ds=load_dataset("akariasai/PopQA",split="test")
def aliases(r):
    a=r["possible_answers"]
    if isinstance(a,str):
        try: a=json.loads(a)
        except: a=[a]
    return a
ITEMS=[{"q":str(r["question"]),"gold":aliases(r),"rel":str(r.get("prop","na"))} for r in ds]
REL=defaultdict(list)
for it in ITEMS:
    for a in it["gold"]: REL[it["rel"]].append(a)
np.random.default_rng(0).shuffle(ITEMS); ITEMS=ITEMS[:N_ITEMS]
print("items per model:",len(ITEMS))

## 1. Run P1/P6/P3/P8 on all models

In [ ]:
PAPER={}
for name in MODELS:
    print("="*70); print(name,flush=True)
    try:
        ctx=mr.make_ctx(name,DEVICE,DTYPE,ITEMS,REL,QA,TEMPLATES)
        out={}
        # each experiment is isolated: one failing does NOT discard the others
        for key,fnc in [
            ("P1_single", lambda: mr.exp_single_token_robustness(ctx, max_items=CAPS["single"])),
            ("P6_hard",   lambda: mr.exp_harddecoy_headline(ctx, max_items=CAPS["hard"])),
            ("P3_dose",   lambda: mr.exp_calibrated_doseresponse(ctx, max_items=CAPS["dose"])),
            ("P8_seq",    lambda: mr.exp_sequence_sanity(ctx, max_items=CAPS["seq"])),
            ("P4_hetero", lambda: mr.exp_intervention_heterogeneity(ctx, max_items=CAPS["dose"], target="p50")),
        ]:
            try:
                out[key]=fnc(); print(f"  {key} done",flush=True)
            except Exception as e:
                import traceback; traceback.print_exc()
                out[key]={"status":"error","error":f"{type(e).__name__}: {e}"}
                print(f"  {key} FAILED: {type(e).__name__}: {str(e)[:140]}",flush=True)
        PAPER[name]=out
    except Exception as e:
        import traceback; traceback.print_exc(); print(f"  -> {type(e).__name__}: {str(e)[:160]}")
        PAPER[name]={"status":"error","error":f"{type(e).__name__}: {e}"}
    finally:
        try: mr.free_ctx(ctx); del ctx
        except Exception: pass
        torch.cuda.empty_cache(); gc.collect()
json.dump(PAPER,open("paper_completion.json","w"),indent=2,default=float)
print("\nsucceeded:",len([k for k,v in PAPER.items() if "status" not in v]),"/",len(PAPER))

## 2. P1 — single-token / prefix robustness (protects 'fact-specific')

In [ ]:
rows=[{"model":nm.split("/")[-1],"n":v["P1_single"]["n"],
        "frac_single":round(v["P1_single"]["frac_single_token"],3),
        "frac_prefix_unique":round(v["P1_single"]["frac_prefix_unique"],3),
        "rho_all":round(v["P1_single"]["rho_int_all"],3),
        "rho_single":round(v["P1_single"]["rho_int_single"],3),
        "rho_unique":round(v["P1_single"]["rho_int_unique"],3),
        "rho_single&unique":round(v["P1_single"]["rho_int_single_and_unique"],3)}
       for nm,v in PAPER.items() if "status" not in v and "status" not in v.get("P1_single",{})]
print(pd.DataFrame(rows).to_string(index=False))
print("\nif rho stays similar on single-token and prefix-unique subsets, the dissociation is not an")
print("artifact of multi-token answers or prefix ambiguity -> the fact-specific claim holds.")

## 3. P6 — hard-decoy headline table

In [ ]:
rows=[{"model":nm.split("/")[-1],"n":v["P6_hard"]["n"],
        "readable_mean":round(v["P6_hard"]["readable_mean"],3),
        "readable_hard":round(v["P6_hard"]["readable_hard"],3),
        "nottop_mean":round(v["P6_hard"]["nottop_rate_mean_readable"],3),
        "nottop_hard":round(v["P6_hard"]["nottop_rate_hard_readable"],3)}
       for nm,v in PAPER.items() if "status" not in v and "status" not in v.get("P6_hard",{})]
print(pd.DataFrame(rows).to_string(index=False))
print("\nlead with readable_hard (answer out-reads the BEST same-type decoy) as the conservative")
print("headline; nottop rate ~1 means readable answers still lose the output.")

## 4. P3 — calibrated dose-response (answer-up / alternative-down / both / random)

In [ ]:
for nm,v in PAPER.items():
    if "status" in v or "status" in v.get("P3_dose",{}): continue
    d=v["P3_dose"]
    if d.get("n",0)==0: print(nm,"-> no successes"); continue
    print("###",nm.split("/")[-1],f"(n_fail={d['n']}, n_success={d['n_success']})")
    print("   targets:", {k:round(val,2) for k,val in d["targets"].items()})
    for inter in ["answer_up","alternative_down","both","random"]:
        cur=d["curves"][inter]
        print(f"   {inter:16s}: "+"  ".join(f"{t}={cur[t]:.2f}" for t in ["p25","p50","p75","mean"]))

## 5. P8 — sequence-level sanity (first-token recovery -> full correct answer)

In [ ]:
rows=[{"model":nm.split("/")[-1],"n":v["P8_seq"]["n"],"target":round(v["P8_seq"]["target"],2),
        "first_tok_recov":round(v["P8_seq"]["first_token_recovered_rate"],3),
        "full_correct":round(v["P8_seq"]["full_answer_correct_rate"],3),
        "full|first":round(v["P8_seq"]["conditional_full_given_first"],3)}
       for nm,v in PAPER.items() if "status" not in v and "status" not in v.get("P8_seq",{})]
print(pd.DataFrame(rows).to_string(index=False))
print("\nfull|first = P(full answer correct | first token recovered). High => first-token recovery")
print("propagates to the correct full answer (the first-token analysis is meaningful, not token-local).")

## 5b. P4 — per-model intervention heterogeneity vs trajectory category
Ties recovery to trajectory structure. The headline case: a strongly alternative-dominant model where
answer-up alone fails but `both` works (positive synergy). Explains Qwen2.5-3B as a real finding.

In [ ]:
rows=[{"model":nm.split("/")[-1],"n":v["P4_hetero"]["n"],
        "alt_dominant":round(v["P4_hetero"]["alternative_dominant_rate"],3),
        "answer_up":round(v["P4_hetero"]["answer_up_recovery"],3),
        "alt_down":round(v["P4_hetero"]["alternative_down_recovery"],3),
        "both":round(v["P4_hetero"]["both_recovery"],3),
        "random":round(v["P4_hetero"]["random_recovery"],3),
        "synergy":round(v["P4_hetero"]["synergy"],3)}
       for nm,v in PAPER.items() if "status" not in v and "status" not in v.get("P4_hetero",{}) and v["P4_hetero"].get("n",0)]
print(pd.DataFrame(rows).to_string(index=False))
print("\npositive synergy (both > max(answer_up, alt_down)) in a strongly alternative-dominant model")
print("is the Qwen2.5-3B finding: suppressing the dominant competitor is necessary alongside raising")
print("the answer. answer_up alone suffices where the model is less competitor-dominant.")

## 6. P2 — OLMo training trajectory (separate; loads checkpoints)
Four measures per checkpoint with a fixed (final-checkpoint) reference frequency direction.
The two-effect reading: `dir_freq` established early and ~flat (co-evolution builds directionality);
`norm_freq` decays over training (weight decay suppresses norm-frequency coupling); `spectral_anisotropy`
tracked independently of the frequency direction.

In [ ]:
try:
    olmo_rows=olr.run_olmo_trajectory(
        model_id="allenai/OLMo-1B-hf", device=DEVICE, dtype=DTYPE,
        want_steps=(500,1000,2000,5000,10000,20000,50000,100000,200000,400000,738000))
    df=pd.DataFrame(olmo_rows)[["step","norm_freq","dir_freq","base_freq","effective_rank","spectral_anisotropy"]]
    print(df.round(3).to_string(index=False))
    # Spearman of each measure vs step (with and without earliest point)
    import mech_core as mc
    steps=df["step"].values
    for col in ["norm_freq","dir_freq","spectral_anisotropy"]:
        rho_all=mc.spearman(steps,df[col].values); rho_drop=mc.spearman(steps[1:],df[col].values[1:])
        print(f"  {col:20s}: Spearman vs step = {rho_all:+.2f} (all), {rho_drop:+.2f} (excl. first)")
    json.dump(olmo_rows,open("olmo_trajectory.json","w"),indent=2,default=float)
    print("\nexpected: norm_freq Spearman strongly negative (decays); dir_freq ~flat (early + stable).")
except Exception as e:
    import traceback; traceback.print_exc()
    print("OLMo run failed (needs network access to allenai/OLMo-1B-hf revisions):", type(e).__name__, str(e)[:160])

## Notes
- **P1** protects the fact-specific claim: report `rho_single` and `rho_unique` alongside `rho_all`.
- **P6**: lead with `readable_hard` as the conservative headline; keep `readable_mean` for context.
- **P3**: the dose-response curves turn the intervention into a calibrated counterfactual; report
  answer-up rising with the target, alternative-down weak alone, both best, random ~0.
- **P8**: `full|first` is the key number; it certifies the first-token analysis transfers to full
  generation. (The generation-time patch is approximate -- a fixed additive nudge along the answer
  direction at the last layer -- so read it as a directional sanity check, not an exact intervention.)
- **P2 (OLMo)**: report Spearman-vs-step for norm_freq (should decay) and dir_freq (should be flat),
  with and without the earliest checkpoint. This is the training-dynamics defense.
- Raise `N_ITEMS`/`CAPS` for final numbers.